In [1]:
from op_patterns import *
from perturb_eval import *
from sympy.core.sympify import sympify
from sympy.core.singleton import S
from sympy.core.symbol import symbols, Symbol, Dummy
from sympy.core.add import Add
from sympy.core.mul import Mul
from sympy.core.power import Pow
from sympy.functions.elementary.trigonometric import cos, sin 
from sympy.functions.elementary.hyperbolic import cosh, sinh

In [2]:
from parametric import TWMSolver as TWM
solver = TWM()
s0 = solver.sol_expr(0, 1)
s0

a_0*(1 + Σ(1 + N_(a_1) + N_(a_2))*N_(a_0)**(-1)/4 - Σ(1 + N_(a_1) + N_(a_2))*N_(a_0)**(-1)*cosh(2*_t*χ*sqrt(N_(a_0)))/4) + Σ(D(a_1)*D(a_2))*BP(a_0; 2)*(-_t*I*χ*N_(a_0)**(-1)/2 + I*N_(a_0)**(-3/2)*sinh(2*_t*χ*sqrt(N_(a_0)))/4) + Σ(a_1*a_2)*(-_t*I*χ/2 - I*N_(a_0)**(-1/2)*sinh(2*_t*χ*sqrt(N_(a_0)))/4)

In [3]:
solver.sol_expr(1, 2)

(a_0*(-_t*I*χ*N_(a_0)**(-1)*cosh(_t*χ*sqrt(N_(a_0)))*Σ(1 + N_(a_1) + N_(a_2))/4 + I*N_(a_0)**(-3/2)*sinh(_t*χ*sqrt(N_(a_0)))*Σ(1 + N_(a_1) + N_(a_2))/4 + I*N_(a_0)**(-3/2)*sinh(3*_t*χ*sqrt(N_(a_0)))*Σ(1 + N_(a_1) + N_(a_2))/16 - I*N_(a_0)**(-3/2)*cosh(_t*χ*sqrt(N_(a_0)))*Σ(1 + N_(a_1) + N_(a_2))/16 - I*N_(a_0)**(-1/2)*sinh(_t*χ*sqrt(N_(a_0)))) + Σ(D(a_1)*D(a_2))*BP(a_0; 2)*(-_t*χ*N_(a_0)**(-3/2)*sinh(_t*χ*sqrt(N_(a_0)))/4 + N_(a_0)**(-2)*sinh(_t*χ*sqrt(N_(a_0)))/16 + N_(a_0)**(-2)*cosh(3*_t*χ*sqrt(N_(a_0)))/16) + Σ(a_1*a_2)*(-_t*χ*N_(a_0)**(-1/2)*sinh(_t*χ*sqrt(N_(a_0)))/4 + 7*N_(a_0)**(-1)*sinh(_t*χ*sqrt(N_(a_0)))/16 - N_(a_0)**(-1)*cosh(3*_t*χ*sqrt(N_(a_0)))/16))*Dagger(a_2) + (_t*χ*(2 + Σ(1 + N_(a_1) + N_(a_2)))*N_(a_0)**(-1/2)*sinh(_t*χ*sqrt(N_(a_0)))/4 + cosh(_t*χ*sqrt(N_(a_0))) - N_(a_0)**(-1)*sinh(_t*χ*sqrt(N_(a_0)))*(8 + Σ(1 + N_(a_1) + N_(a_2)))/16 + Σ(D(a_1)*D(a_2))*a_0*(-_t*I*χ*N_(a_0)**(-1)*cosh(_t*χ*sqrt(N_(a_0)))/4 + I*N_(a_0)**(-3/2)*sinh(_t*χ*sqrt(N_(a_0)))/4 + I*N_(a_0

In [ ]:
# Use the methods for TWM to calculate the FWM parametric solution and pump depletion
from sympy.core.function import expand_mul
from parametric import (I, Dagger, BosonOp, exp,
                        t_, nlc, a_i, a_p, a_s, pm, _g,
                        MultimodeNum as MMN, MultimodeNum2 as MMN2, MultimodeCOP as MMC)
from perturb_eval.dsolve_pm import dsd1, dsd2

# def other_d(a: BosonOp):
#     if a.name == a_s.name:
#         return BosonOp('a_2', not a.is_annihilation)
#     return BosonOp('a_1', not a.is_annihilation)
# # tuple drop zeroes
# def tdz(i, exps):
#     lz=False
#     new_exps = []
#     for e in exps:
#         if lz:
#             new_exps[-1] += e
#             lz = False
#         elif e == 0:
#             if not new_exps:
#                 i = not i
#             else:
#                 lz = True
#         else:
#             new_exps.append(e)
#     if new_exps:
#         return (i, *new_exps)
#     return (None,)
# def btswap(a: BosonOp, i, exps, to_left=True):
#     yield S.One, a, exps
#     if to_left:
#         for j, e in enumerate(exps):
#             if i ^ (j & 1) != a.is_annihilation:
#                 for s, b, nes in btswap(other_d(a), i, exps[:j], to_left):
#                     yield S.NegativeOne**int(a.is_annihilation)*s*e, b, (*nes, e-1, *exps[j+1:])
#     else:
#         for j, e in enumerate(exps):
#             ic = i ^ (j & 1)
#             if ic != a.is_annihilation:
#                 for s, b, nes in btswap(other_d(a), not ic, exps[j+1:], to_left):
#                     yield S.NegativeOne**int(ic)*e*s, b, (*exps[:j], e-1, *nes)
# def bpswap(a: BosonOp, p: PF, s, to_left=True):
#     def mmn_shift(mmn):
#         if mmn.name == a.name:
#             if a.is_annihilation ^ to_left:
#                 return mmn+1
#             return mmn-1
#         return mmn
#     def mmn2_shift(mmn2):
#         if a.is_annihilation ^ to_left:
#             return mmn2+1
#         return mmn2-1
#     return p.n_apply(lambda e: s*e.replace(MMN, mmn_shift).replace(MMN2, mmn2_shift))
# def bdswap(a: BosonOp, d, to_left=True):
#     new_dict = {a: dict(), other_d(a): dict()}
#     for t, p in d.items():
#         i, *exps = t
#         for s, b, nes in btswap(a, i, exps, to_left):
#             key = tdz(i, nes)
#             if key in new_dict[b]:
#                 new_dict[b][tdz(i, nes)] = PF.add(new_dict[b][tdz(i, nes)], bpswap(b, p, s, to_left))
#             else:
#                 new_dict[b][tdz(i, nes)] = bpswap(b, p, s, to_left)
#     return new_dict

def combine_exp(expr):
    terms = []
    for term in Add.make_args(expand_mul(expr, deep=False)):
        if isinstance(term, Mul):
            e = []
            ne = []
            for arg in expr.args:
                if isinstance(arg, exp):
                    e.append(arg.args[0])
                else:
                    ne.append(arg)
            terms.append(Mul(*ne, exp(Add(*e))))
        else:
            terms.append(term)
    return Add(terms)



In [ ]:
solver_pm = TWM(True, True)
n_p = BosonNum(a_p.name)
hpm = -I*pm*S.Half + I*nlc*n_p  # Half phase mismatch
gc = S.Half*(pm * (pm + nlc*n_p))**S.Half
pe = PE({a_p.name: -S.Half, nlc: S.One})
pe.register(MMN)
pe.register(MMN2)
pe.register(MMC)
fwm1 = ({(None,): PF({((a_p,), ()): S.One})},
        {a_s: {(None,): PF({((), ()): dsd2(gc*t_, dict(), True, S.One, hpm/gc)})},
         Dagger(a_i): {(None,): PF({((a_p**2,), ()): dsd2(gc*t_, dict(), True, S.Zero,
                                                          -I*nlc/gc)})}},
        {Dagger(a_s): {(None,): PF({((a_p**2,), ()): dsd2(gc*t_, dict(), True, S.Zero,
                                                          -I*nlc/gc)})},
         a_i: {(None,): PF({((), ()): dsd2(gc*t_, dict(), True, S.One, hpm/gc)})}})

prod_10 = {a_s: dict(), Dagger(a_i): dict()}
d0 = solver.d_adjoint(fwm1[0])
for b, e in fwm1[1].items():
    for a, d in solver_pm.bdswap(b, d0, False).items():
        solver_pm.d_mul(prod_10[a], e, d)

dd2 = solver_pm.e_mul(prod_10, fwm1[2])
fwm2p = dict()
for k, v in dd2.items():
    rhs = pe.expand_pf(v.n_apply(lambda t: -2*I*nlc*t.subs({t_: _g/gc})/gc), 1)
    s0 = rhs.n_apply(lambda t: solver_pm.n_simp(dsd1(_g, solver_pm.p2s_group(t), subs=gc*t_)))
    fwm2p[k] = PF.add(fwm1[0][(None,)], s0) if len(k) == 1 else s0
solver.to_expr(fwm2p)*exp(-I * nlc * n_p)

(a_0*(2*_t*I*χ**2*N_(a_0)*(Δk + χ*N_(a_0))**(-1)*(-Δk + 2*χ*N_(a_0))*Σ(1 + N_(a_1) + N_(a_2))/Δk + 1 - 2*I*χ**2*N_(a_0)*(Δk*(Δk + χ*N_(a_0)))**(-1/2)*(Δk + χ*N_(a_0))**(-1)*(-Δk + 2*χ*N_(a_0))*sinh(_t*sqrt(Δk*(Δk + χ*N_(a_0))))*Σ(1 + N_(a_1) + N_(a_2))/Δk - 2*χ**2*N_(a_0)*(Δk + χ*N_(a_0))**(-1)*cosh(_t*sqrt(Δk*(Δk + χ*N_(a_0))))*Σ(1 + N_(a_1) + N_(a_2))/Δk + 2*χ**2*N_(a_0)*(Δk + χ*N_(a_0))**(-1)*Σ(1 + N_(a_1) + N_(a_2))/Δk) + Σ(D(a_1)*D(a_2))*BP(a_0; 3)*(-4*_t*I*χ**3*N_(a_0)*(Δk + χ*N_(a_0))**(-1)/Δk + 4*I*χ**3*N_(a_0)*(Δk*(Δk + χ*N_(a_0)))**(-3/2)*sinh(_t*sqrt(Δk*(Δk + χ*N_(a_0))))) + Σ(a_1*a_2)*Dagger(a_0)*(-_t*I*χ*(Δk + χ*N_(a_0))**(-1)*(2*Δk**2 - 3*Δk*χ*N_(a_0) + 4*χ**2*N_(a_0)**2)/Δk + I*χ**2*N_(a_0)*(Δk*(Δk + χ*N_(a_0)))**(-1/2)*(Δk + χ*N_(a_0))**(-1)*(-5*Δk + 4*χ*N_(a_0))*sinh(_t*sqrt(Δk*(Δk + χ*N_(a_0))))/Δk - 2*χ*(Δk + χ*N_(a_0))**(-1)*(-Δk + 2*χ*N_(a_0))/Δk + 2*χ*(Δk + χ*N_(a_0))**(-1)*(-Δk + 2*χ*N_(a_0))*cosh(_t*sqrt(Δk*(Δk + χ*N_(a_0))))/Δk))*exp(-I*χ*N_(a_0))